In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional, Union, Dict, Any, Callable
from itertools import combinations
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
gt = get_backtest('6148bef0b6749052fd1f25ff724051d0')#修改成你的策略id
data = gt.get_results()
df = pd.DataFrame(data)
df['daily_returns'] = (1 + df['returns']) / (1 + df['returns'].shift(1)) - 1
bt_ret = df['daily_returns'][1:]


In [ ]:
def _check_sample_size(returns: np.ndarray, min_days: int = 500) -> None:
    """检查样本量是否足够，不足则给出警告"""
    if len(returns) < min_days:
        print(f"⚠️ 警告：样本量仅 {len(returns)} 天，少于建议的 {min_days} 天，部分检验可靠性降低")


def _safe_sharpe(returns: np.ndarray, annual_factor: float = np.sqrt(252)) -> float:
    """安全计算夏普比率（避免除零）"""
    if len(returns) == 0:
        return 0.0
    mu = np.mean(returns)
    sigma = np.std(returns, ddof=1)
    if sigma < 1e-8:
        return 0.0
    return mu / sigma * annual_factor


def _annual_return(returns: np.ndarray, periods: int = 252) -> float:
    """计算年化收益率"""
    if len(returns) == 0:
        return 0.0
    total = np.prod(1 + returns) - 1
    return (1 + total) ** (periods / len(returns)) - 1


def _max_drawdown(returns: np.ndarray) -> float:
    """计算最大回撤"""
    if len(returns) == 0:
        return 0.0
    cum = np.cumprod(1 + returns)
    peak = np.maximum.accumulate(cum)
    return np.min(cum / peak - 1)

In [ ]:
# ========================= 1. 时间稳定性检验 =========================

def check_temporal_stability(
    returns: np.ndarray,
    window: int = 252,
    step: int = 21
) -> Dict[str, Any]:
    """
    使用滚动窗口夏普比率评估时间稳定性。
    返回：
        - rolling_sharpes: 滚动夏普序列
        - mean_sharpe, std_sharpe, cv: 描述统计
        - p_value: Kruskal-Wallis检验p值
        - is_stable: 综合稳定判断（cv<1.0 且 p>0.05）
    """
    returns = np.asarray(returns)
    n = len(returns)
    if n < window:
        return {"is_stable": False, "message": "样本量小于窗口长度", "p_value": 1.0}

    rolling_sharpes = []
    for i in range(0, n - window + 1, step):
        seg = returns[i:i+window]
        rolling_sharpes.append(_safe_sharpe(seg))

    if len(rolling_sharpes) < 2:
        return {"is_stable": True, "message": "滚动窗口数量不足", "p_value": 1.0}

    mean_sharpe = np.mean(rolling_sharpes)
    std_sharpe = np.std(rolling_sharpes)
    cv = std_sharpe / abs(mean_sharpe) if mean_sharpe != 0 else np.inf

    # Kruskal-Wallis 检验：将滚动夏普按时序分为3组
    n_groups = min(3, len(rolling_sharpes) // 2)
    if n_groups >= 2:
        group_size = len(rolling_sharpes) // n_groups
        groups = [rolling_sharpes[i*group_size:(i+1)*group_size] for i in range(n_groups-1)]
        groups.append(rolling_sharpes[(n_groups-1)*group_size:])
        _, p_value = stats.kruskal(*groups)
    else:
        p_value = 1.0

    stable = (cv < 1.0) and (p_value > 0.05)

    print(f"滚动夏普均值: {mean_sharpe:.2f}, 标准差: {std_sharpe:.2f}")
    print(f"变异系数: {cv:.2f} (越小越稳定)")
    print(f"Kruskal-Wallis p值: {p_value:.4f} (>0.05表示各时期表现无显著差异)")
    print(f"稳定性结论: {'✅ 稳定' if stable else '⚠️ 不稳定'}")

    return {
        "rolling_sharpes": rolling_sharpes,
        "mean_sharpe": mean_sharpe,
        "std_sharpe": std_sharpe,
        "cv": cv,
        "p_value": p_value,
        "is_stable": stable
    }


In [ ]:
# ========================= 2. 游程检验（支持零收益） =========================

def runs_test(
    returns: np.ndarray,
    zero_treatment: str = "separate",
    random_state: int = 42
) -> Dict[str, Any]:
    """
    游程检验：检验收益符号序列是否随机。
    zero_treatment: 'separate'(零单独分类), 'ignore'(删除零), 'sign'(零归入正或负)
    """
    returns = np.asarray(returns)
    if zero_treatment == "separate":
        signs = np.where(returns > 0, 1, np.where(returns < 0, -1, 0))
    elif zero_treatment == "ignore":
        signs = np.sign(returns)
        signs = signs[signs != 0]
    elif zero_treatment == "sign":
        signs = np.sign(returns)
    else:
        raise ValueError("zero_treatment must be 'separate', 'ignore', or 'sign'")

    n = len(signs)
    if n == 0:
        return {"is_random": False, "message": "无有效数据", "p_value": 1.0}

    unique, counts = np.unique(signs, return_counts=True)
    counts_dict = dict(zip(unique, counts))
    n_pos = counts_dict.get(1, 0)
    n_neg = counts_dict.get(-1, 0)
    n_zero = counts_dict.get(0, 0)

    runs = 1
    for i in range(1, n):
        if signs[i] != signs[i-1]:
            runs += 1

    rng = np.random.RandomState(random_state)

    if zero_treatment == "separate" and n_zero > 0:
        # 蒙特卡洛置换检验
        n_perm = 2000
        runs_perm = []
        for _ in range(n_perm):
            perm = rng.permutation(signs)
            r = 1 + np.sum(perm[:-1] != perm[1:])
            runs_perm.append(r)
        runs_perm = np.array(runs_perm)
        p_value = np.mean(runs_perm >= runs) if runs >= np.mean(runs_perm) else np.mean(runs_perm <= runs)
        p_value = min(p_value, 1 - p_value) * 2
        expected_runs = np.mean(runs_perm)
        z_stat = (runs - expected_runs) / np.std(runs_perm)
    else:
        n1, n2 = n_pos, n_neg
        if n1 == 0 or n2 == 0:
            return {"is_random": True, "message": "序列全同号", "runs": 1, "p_value": 1.0}
        expected_runs = (2.0 * n1 * n2) / n + 1.0
        numerator = 2.0 * n1 * n2 * (2.0 * n1 * n2 - n)
        denominator = n * n * (n - 1.0)
        var_runs = numerator / denominator if denominator > 0 else 0.0
        std_runs = np.sqrt(var_runs)
        if std_runs > 0:
            z_stat = (runs - expected_runs) / std_runs
            p_value = 2.0 * (1.0 - stats.norm.cdf(abs(z_stat)))
        else:
            z_stat, p_value = 0.0, 1.0

    print(f"样本量: {n}, 正: {n_pos}, 负: {n_neg}, 零: {n_zero if zero_treatment=='separate' else 0}")
    print(f"实际游程数: {runs}, 期望游程数: {expected_runs:.2f}")
    print(f"Z统计量: {z_stat:.2f}, p值: {p_value:.4f}")

    return {
        "runs": runs,
        "expected_runs": expected_runs,
        "z_stat": z_stat,
        "p_value": p_value,
        "is_random": p_value > 0.05
    }

In [ ]:
# ========================= 3. 方差比率检验（修正版） =========================

def variance_ratio_test(
    returns: np.ndarray,
    q_list: List[int] = [2, 5, 10]
) -> Dict[str, Any]:
    """
    方差比率检验（Lo & MacKinlay 1988，异方差稳健标准误，向量化加速）
    返回：
        - by_q: 各尺度详细结果
        - min_p_value: 所有尺度中的最小p值
        - is_random_walk: 最小p值 > 0.05 则认为符合随机游走
    """
    returns = np.asarray(returns)
    n = len(returns)
    mu = np.mean(returns)
    y = returns - mu
    y2 = y ** 2
    sum_y2 = np.sum(y2)
    var_1 = sum_y2 / n

    results = {}
    for q in q_list:
        if n <= q:
            continue
        # q期累积收益
        y_q = np.array([np.sum(y[i:i+q]) for i in range(n - q + 1)])
        var_q = np.sum(y_q**2) / (n - q + 1) / q
        vr = var_q / var_1 if var_1 > 0 else 1.0

        # 异方差稳健标准误（向量化计算 delta）
        delta = np.zeros(q-1)
        for j in range(1, q):
            w = 2 * (q - j) / q
            num = np.dot(y2[j:], y2[:n-j])
            denom = sum_y2 ** 2
            delta[j-1] = w * (num / denom) if denom > 0 else 0.0
        se = np.sqrt(2 * np.sum(delta))

        z_stat = (vr - 1) / se if se > 0 else 0.0
        p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

        results[q] = {
            "vr": vr,
            "z_stat": z_stat,
            "p_value": p_value,
            "is_random_walk": p_value > 0.05
        }
        print(f"q={q}: VR={vr:.4f}, Z={z_stat:.2f}, p={p_value:.4f} -> {'随机游走' if p_value>0.05 else '非随机'}")

    p_values = [res["p_value"] for res in results.values()]
    min_p = min(p_values) if p_values else 1.0
    return {
        "by_q": results,
        "min_p_value": min_p,
        "is_random_walk": min_p > 0.05
    }

In [ ]:
# ========================= 4. Chow 结构断点检验（仅截距模型） =========================

def chow_test(
    returns: np.ndarray,
    break_point: Optional[int] = None
) -> Dict[str, Any]:
    """
    Chow检验检测前后段均值是否存在显著差异（仅截距模型）。
    """
    returns = np.asarray(returns)
    n = len(returns)
    if break_point is None:
        break_point = n // 2
    if break_point < 2 or break_point > n-2:
        return {"has_break": False, "message": "断点位置无效", "p_value": 1.0}

    y = returns.reshape(-1, 1)
    X = np.ones((n, 1))  # 仅截距

    # 全样本
    beta_full = np.linalg.lstsq(X, y, rcond=None)[0]
    resid_full = y - X @ beta_full
    rss_full = np.sum(resid_full**2)

    # 前段
    X1, y1 = X[:break_point], y[:break_point]
    beta1 = np.linalg.lstsq(X1, y1, rcond=None)[0]
    rss1 = np.sum((y1 - X1 @ beta1)**2)

    # 后段
    X2, y2 = X[break_point:], y[break_point:]
    beta2 = np.linalg.lstsq(X2, y2, rcond=None)[0]
    rss2 = np.sum((y2 - X2 @ beta2)**2)

    k = 1  # 参数个数
    numerator = (rss_full - (rss1 + rss2)) / k
    denominator = (rss1 + rss2) / (n - 2*k)
    f_stat = numerator / denominator if denominator > 0 else 0.0
    p_value = 1 - stats.f.cdf(f_stat, k, n - 2*k)
    has_break = p_value < 0.05

    print(f"断点位置: {break_point}, 前段夏普: {_safe_sharpe(returns[:break_point]):.2f}, 后段夏普: {_safe_sharpe(returns[break_point:]):.2f}")
    print(f"Chow F={f_stat:.3f}, p={p_value:.4f} -> {'存在断点' if has_break else '无明显断点'}")

    return {"f_stat": f_stat, "p_value": p_value, "has_break": has_break}


In [ ]:
# ========================= 5. 块自助法敏感性 =========================

def block_bootstrap_sensitivity(
    returns: np.ndarray,
    n_bootstrap: int = 1000,
    block_size: Optional[int] = None,
    random_state: int = 42
) -> Dict[str, Any]:
    """块自助法评估原始夏普是否极端"""
    returns = np.asarray(returns)
    n = len(returns)
    if block_size is None:
        block_size = int(round(n ** (1/3)))
    block_size = max(1, min(block_size, n // 2))

    rng = np.random.RandomState(random_state)
    original_sharpe = _safe_sharpe(returns)

    bootstrap_sharpes = []
    for _ in range(n_bootstrap):
        n_blocks = int(np.ceil(n / block_size))
        blocks = []
        for _ in range(n_blocks):
            start = rng.randint(0, n - block_size + 1)
            blocks.append(returns[start:start+block_size])
        sample = np.concatenate(blocks)[:n]
        bootstrap_sharpes.append(_safe_sharpe(sample))

    bootstrap_sharpes = np.array(bootstrap_sharpes)
    percentile = stats.percentileofscore(bootstrap_sharpes, original_sharpe)
    is_extreme = (percentile < 2.5) or (percentile > 97.5)

    print(f"原始夏普: {original_sharpe:.2f}, Bootstrap中位数: {np.median(bootstrap_sharpes):.2f}, 百分位: {percentile:.1f}%")
    print(f"结论: {'极端' if is_extreme else '未发现极端异常'}")

    return {
        "original_sharpe": original_sharpe,
        "bootstrap_sharpes": bootstrap_sharpes,
        "percentile": percentile,
        "is_extreme": is_extreme
    }

In [ ]:
# ========================= 6. 交易成本敏感性 =========================

def transaction_cost_sensitivity(
    returns: np.ndarray,
    turnover_series: Optional[np.ndarray] = None,
    cost_bps_range: Tuple[float, float] = (0, 50),
    n_points: int = 11
) -> Dict[str, Any]:
    """成本敏感性分析"""
    returns = np.asarray(returns)
    if turnover_series is not None:
        turnover_series = np.asarray(turnover_series)
        avg_turnover = np.mean(turnover_series)
    else:
        print("⚠️ 未提供换手率，使用默认日均0.03")
        avg_turnover = 0.03
        turnover_series = np.full_like(returns, avg_turnover)

    cost_bps = np.linspace(cost_bps_range[0], cost_bps_range[1], n_points)
    net_sharpes = []
    original_sharpe = _safe_sharpe(returns)

    for c_bps in cost_bps:
        cost_per_trade = c_bps / 10000.0
        daily_cost = turnover_series * 2 * cost_per_trade
        net_returns = returns - daily_cost
        net_sharpes.append(_safe_sharpe(net_returns))

    idx_10bps = np.argmin(np.abs(cost_bps - 10))
    sharpe_10bps = net_sharpes[idx_10bps]
    is_robust = (sharpe_10bps > 0) and (sharpe_10bps > original_sharpe * 0.5)

    print(f"原始夏普: {original_sharpe:.2f}, 10bps后: {sharpe_10bps:.2f}, 50bps后: {net_sharpes[-1]:.2f}")
    print(f"成本鲁棒性: {'✅ 良好' if is_robust else '⚠️ 敏感'}")

    return {
        "cost_bps": cost_bps,
        "net_sharpes": net_sharpes,
        "is_cost_robust": is_robust,
        "sharpe_10bps": sharpe_10bps
    }


In [ ]:
# ========================= 7. 滚动交叉验证 =========================

def rolling_cross_validation(
    returns: np.ndarray,
    train_window: int = 504,
    test_window: int = 63,
    step: int = 21,
    strategy_func: Optional[Callable[[np.ndarray], np.ndarray]] = None
) -> Dict[str, Any]:
    """
    滚动样本外验证。
    若提供 strategy_func，则应接受训练期收益，返回测试期预测收益序列。
    """
    returns = np.asarray(returns)
    n = len(returns)
    if n < train_window + test_window:
        return {"message": "样本量不足", "is_overfit": False, "decay": 0.0}

    oos_sharpes = []
    for start in range(0, n - train_window - test_window + 1, step):
        train_end = start + train_window
        test_end = train_end + test_window
        train_returns = returns[start:train_end]
        test_returns = returns[train_end:test_end]

        if strategy_func is not None:
            try:
                pred_returns = strategy_func(train_returns)
                if len(pred_returns) != test_window:
                    pred_returns = pred_returns[:test_window]
                oos_sharpe = _safe_sharpe(pred_returns)
            except Exception:
                continue
        else:
            oos_sharpe = _safe_sharpe(test_returns)
        oos_sharpes.append(oos_sharpe)

    if len(oos_sharpes) == 0:
        return {"message": "无有效窗口", "is_overfit": False, "decay": 0.0}

    mean_oos = np.mean(oos_sharpes)
    std_oos = np.std(oos_sharpes)
    full_sharpe = _safe_sharpe(returns)
    decay = full_sharpe - mean_oos

    print(f"全样本夏普: {full_sharpe:.2f}")
    print(f"样本外夏普均值: {mean_oos:.2f} ± {std_oos:.2f}")
    print(f"样本外衰减: {decay:.2f}")

    return {
        "oos_sharpes": oos_sharpes,
        "mean_oos": mean_oos,
        "std_oos": std_oos,
        "decay": decay,
        "is_overfit": decay > 0.5
    }

In [ ]:
# ========================= 8. 近似PBO（强化警告） =========================

def approximate_pbo(
    returns: np.ndarray,
    n_trials: int = 100,
    block_size: Optional[int] = None,
    random_state: int = 42
) -> Dict[str, Any]:
    """
    基于块自助法的近似PBO计算。注意：此为近似值，可能低估真实过拟合风险。
    """
    returns = np.asarray(returns)
    n = len(returns)
    if block_size is None:
        block_size = int(round(n ** (1/3)))
    block_size = max(1, min(block_size, n // 4))

    rng = np.random.RandomState(random_state)

    trials_matrix = np.zeros((n, n_trials))
    for i in range(n_trials):
        n_blocks = int(np.ceil(n / block_size))
        blocks = []
        for _ in range(n_blocks):
            start = rng.randint(0, n - block_size + 1)
            blocks.append(returns[start:start+block_size])
        sample = np.concatenate(blocks)[:n]
        trials_matrix[:, i] = sample

    sharpes = np.array([_safe_sharpe(trials_matrix[:, i]) for i in range(n_trials)])
    valid = np.isfinite(sharpes)
    sharpes = sharpes[valid]
    trials_matrix = trials_matrix[:, valid]
    n_trials_valid = trials_matrix.shape[1]

    if n_trials_valid < 10:
        return {"pbo": np.nan, "message": "有效变体数量不足"}

    S = min(8, n // 4)
    if S % 2 != 0:
        S = max(2, S - 1)
    block_len = n // S
    combos = list(combinations(range(S), S//2))

    oos_ranks = []
    for combo in combos:
        is_mask = np.zeros(S, dtype=bool)
        is_mask[list(combo)] = True
        oos_mask = ~is_mask

        is_idx = np.concatenate([np.arange(i*block_len, (i+1)*block_len) for i, m in enumerate(is_mask) if m])
        oos_idx = np.concatenate([np.arange(i*block_len, (i+1)*block_len) for i, m in enumerate(oos_mask) if m])

        is_perf = np.array([_safe_sharpe(trials_matrix[is_idx, j]) for j in range(n_trials_valid)])
        oos_perf = np.array([_safe_sharpe(trials_matrix[oos_idx, j]) for j in range(n_trials_valid)])
        valid_both = np.isfinite(is_perf) & np.isfinite(oos_perf)
        if np.sum(valid_both) < 5:
            continue
        is_perf = is_perf[valid_both]
        oos_perf = oos_perf[valid_both]
        best_idx = np.argmax(is_perf)
        oos_rank = np.mean(oos_perf >= oos_perf[best_idx])
        oos_ranks.append(oos_rank)

    if len(oos_ranks) == 0:
        return {"pbo": np.nan, "message": "无有效组合"}

    oos_ranks = np.array(oos_ranks)
    pbo = np.mean(oos_ranks <= 0.5)

    print(f"近似PBO: {pbo:.3f} ({'低' if pbo<0.2 else '中' if pbo<0.4 else '高'}风险)")
    print("⚠️ 注意：此PBO为近似值，可能低估真实过拟合风险。完整评估需提供策略参数搜索矩阵。")

    return {
        "pbo": pbo,
        "oos_ranks": oos_ranks,
        "warning": "近似PBO，可能低估真实过拟合风险"
    }

In [ ]:
# ========================= 9. 实盘对比 =========================

def live_vs_backtest_comparison(
    bt_returns: np.ndarray,
    live_returns: np.ndarray
) -> Dict[str, Any]:
    """对比回测与实盘关键指标"""
    bt = np.asarray(bt_returns)
    live = np.asarray(live_returns)

    bt_sharpe = _safe_sharpe(bt)
    live_sharpe = _safe_sharpe(live)
    bt_ret = _annual_return(bt)
    live_ret = _annual_return(live)
    bt_dd = _max_drawdown(bt)
    live_dd = _max_drawdown(live)

    sharpe_decay = bt_sharpe - live_sharpe
    ks_stat, ks_p = stats.ks_2samp(bt, live)

    print(f"回测夏普: {bt_sharpe:.2f}, 实盘夏普: {live_sharpe:.2f}, 衰减: {sharpe_decay:.2f}")
    print(f"回测年化收益: {bt_ret:.2%}, 实盘年化收益: {live_ret:.2%}")
    print(f"KS检验 p值: {ks_p:.4f} (分布一致性)")

    is_consistent = (ks_p > 0.05) and (sharpe_decay < 1.0)

    return {
        "bt_sharpe": bt_sharpe,
        "live_sharpe": live_sharpe,
        "sharpe_decay": sharpe_decay,
        "bt_ret": bt_ret,
        "live_ret": live_ret,
        "ks_p": ks_p,
        "is_consistent": is_consistent
    }


In [ ]:
def _plot_diagnostics(
    returns: np.ndarray,
    results: Dict[str, Any],
    live_returns: Optional[np.ndarray] = None
) -> None:
    """绘制诊断图表"""
    n_plots = 6 if live_returns is not None else 5
    n_cols = 3
    n_rows = (n_plots + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
    axes = axes.flatten()

    # 图1：累计收益
    ax = axes[0]
    cum = np.cumprod(1 + returns)
    ax.plot(cum, label="回测", color="blue")
    if live_returns is not None:
        live_cum = np.cumprod(1 + live_returns)
        ax.plot(np.arange(len(returns), len(returns)+len(live_returns)),
                live_cum * cum[-1], label="实盘", color="red", linestyle="--")
    ax.set_title("累计收益曲线")
    ax.legend()

    # 图2：滚动夏普
    ax = axes[1]
    if "rolling_sharpes" in results["stability"]:
        rs = results["stability"]["rolling_sharpes"]
        ax.plot(rs, color="green")
        ax.axhline(y=0, linestyle="--", color="gray")
        ax.set_title("滚动窗口夏普比率")
        ax.set_xlabel("窗口序号")

    # 图3：成本敏感性
    ax = axes[2]
    cost_data = results["cost"]
    ax.plot(cost_data["cost_bps"], cost_data["net_sharpes"], marker="o", color="red")
    ax.axhline(y=0, linestyle="--", color="gray")
    ax.set_title("夏普比率 vs 交易成本")
    ax.set_xlabel("成本 (bps)")

    # 图4：样本外夏普分布
    ax = axes[3]
    if "oos_sharpes" in results["cv"] and len(results["cv"]["oos_sharpes"]) > 0:
        ax.hist(results["cv"]["oos_sharpes"], bins=15, color="purple", alpha=0.7)
        ax.axvline(x=results["cv"]["mean_oos"], color="black", linestyle="--", label="均值")
        ax.set_title("样本外夏普分布")
        ax.legend()
    else:
        ax.text(0.5, 0.5, "样本外数据不足", ha="center", va="center")
        ax.set_title("样本外夏普分布")

    # 图5：Bootstrap夏普分布
    ax = axes[4]
    if "bootstrap_sharpes" in results["bootstrap"]:
        ax.hist(results["bootstrap"]["bootstrap_sharpes"], bins=30, color="orange", alpha=0.7)
        ax.axvline(x=results["bootstrap"]["original_sharpe"], color="red", linestyle="--", label="原始")
        ax.set_title("Bootstrap夏普分布")
        ax.legend()

    # 图6：实盘对比
    if live_returns is not None and n_plots >= 6:
        ax = axes[5]
        metrics = ["夏普", "年化收益", "最大回撤"]
        bt_vals = [results["live_compare"]["bt_sharpe"],
                   results["live_compare"]["bt_ret"],
                   _max_drawdown(returns)]
        live_vals = [results["live_compare"]["live_sharpe"],
                     results["live_compare"]["live_ret"],
                     _max_drawdown(live_returns)]
        x = np.arange(len(metrics))
        width = 0.35
        ax.bar(x - width/2, bt_vals, width, label="回测", color="blue")
        ax.bar(x + width/2, live_vals, width, label="实盘", color="red")
        ax.set_xticks(x)
        ax.set_xticklabels(metrics)
        ax.set_title("回测 vs 实盘关键指标")
        ax.legend()

    # 隐藏多余子图
    for i in range(n_plots, len(axes)):
        axes[i].set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
# ========================= 综合诊断（修正版） =========================

def comprehensive_overfitting_check(
    returns: np.ndarray,
    live_returns: Optional[np.ndarray] = None,
    strategy_func: Optional[Callable[[np.ndarray], np.ndarray]] = None,
    strategy_name: str = "Strategy",
    show_plots: bool = True,
    bonferroni_alpha: float = 0.05,
    random_state: int = 42
) -> Dict[str, Any]:
    """
    综合过拟合诊断报告（修正版）

    参数:
        returns: 回测日收益率
        live_returns: 实盘日收益率（可选）
        strategy_func: 自定义策略函数，用于滚动CV
        strategy_name: 策略名称
        show_plots: 是否显示图表
        bonferroni_alpha: 多重检验校正的族错误率
        random_state: 随机种子
    """
    _check_sample_size(returns)

    print(f"\n{'='*60}")
    print(f"过拟合诊断报告: {strategy_name}")
    print(f"回测样本: {len(returns)} 天 ({len(returns)/252:.1f} 年)")
    if live_returns is not None:
        print(f"实盘样本: {len(live_returns)} 天")
    print(f"{'='*60}")

    # 基础统计
    ann_ret = _annual_return(returns)
    vol = np.std(returns) * np.sqrt(252)
    sharpe = ann_ret / vol if vol > 0 else 0
    max_dd = _max_drawdown(returns)

    print(f"\n基础指标:")
    print(f"  年化收益: {ann_ret:.2%}")
    print(f"  年化波动: {vol:.2%}")
    print(f"  夏普比率: {sharpe:.2f}")
    print(f"  最大回撤: {max_dd:.2%}")

    results = {}
    p_values = []  # 仅用于Bonferroni校正的统计检验p值

    # 1. 时间稳定性
    print(f"\n--- 1. 滚动夏普稳定性 ---")
    results["stability"] = check_temporal_stability(returns)
    p_values.append(results["stability"]["p_value"])

    # 2. 游程检验
    print(f"\n--- 2. 游程随机性检验 ---")
    results["runs"] = runs_test(returns, random_state=random_state)
    p_values.append(results["runs"]["p_value"])

    # 3. 方差比率（使用最小p值）
    print(f"\n--- 3. 方差比率检验 ---")
    results["vr"] = variance_ratio_test(returns)
    p_values.append(results["vr"]["min_p_value"])

    # 4. Chow检验
    print(f"\n--- 4. 结构断点检验 ---")
    results["chow"] = chow_test(returns)
    p_values.append(results["chow"]["p_value"])

    # 5. 块自助法
    print(f"\n--- 5. 块自助法敏感性 ---")
    results["bootstrap"] = block_bootstrap_sensitivity(returns, random_state=random_state)

    # 6. 成本敏感性
    print(f"\n--- 6. 交易成本敏感性 ---")
    results["cost"] = transaction_cost_sensitivity(returns)

    # 7. 滚动交叉验证
    print(f"\n--- 7. 滚动样本外验证 ---")
    results["cv"] = rolling_cross_validation(returns, strategy_func=strategy_func)

    # 8. 近似PBO
    print(f"\n--- 8. 近似PBO计算 ---")
    results["approx_pbo"] = approximate_pbo(returns, random_state=random_state)

    # 9. 实盘对比
    if live_returns is not None:
        print(f"\n--- 9. 实盘 vs 回测对比 ---")
        results["live_compare"] = live_vs_backtest_comparison(returns, live_returns)

    # 多重检验校正
    n_tests = len(p_values)
    bonferroni_threshold = bonferroni_alpha / n_tests if n_tests > 0 else bonferroni_alpha
    print(f"\n多重检验校正: {n_tests} 项统计检验, Bonferroni阈值 = {bonferroni_threshold:.4f}")

    # 收集风险信号（区分类型）
    statistical_risks = []
    effect_risks = []

    # 统计显著性风险（使用Bonferroni校正阈值）
    if results["stability"]["p_value"] < bonferroni_threshold:
        statistical_risks.append("时间不稳定")
    if results["runs"]["p_value"] < bonferroni_threshold:
        statistical_risks.append("收益序列非随机")
    if results["vr"]["min_p_value"] < bonferroni_threshold:
        statistical_risks.append("非随机游走")
    if results["chow"]["p_value"] < bonferroni_threshold:
        statistical_risks.append("存在结构断点")

    # 效应量/启发式风险（固定阈值）
    if results["bootstrap"]["is_extreme"]:
        effect_risks.append("夏普对历史路径敏感")
    if not results["cost"]["is_cost_robust"]:
        effect_risks.append("交易成本敏感")
    if results["cv"].get("is_overfit", False):
        effect_risks.append("样本外衰减过大")
    if live_returns is not None and not results.get("live_compare", {}).get("is_consistent", True):
        effect_risks.append("实盘与回测显著偏离")

    # 综合输出
    print(f"\n{'='*60}")
    print("综合评估:")
    all_risks = statistical_risks + effect_risks
    if all_risks:
        print(f"⚠️ 发现 {len(all_risks)} 项风险信号：")
        if statistical_risks:
            print(f"  - 统计显著性风险 ({len(statistical_risks)}项): {', '.join(statistical_risks)}")
        if effect_risks:
            print(f"  - 效应量/启发式风险 ({len(effect_risks)}项): {', '.join(effect_risks)}")
        print("建议：简化策略、增加正则化、延长样本外测试、降低参数维度")
    else:
        print("✅ 未检测到明显过拟合特征，但仍建议进行独立样本外验证")
    print(f"{'='*60}")

    if show_plots:
        _plot_diagnostics(returns, results, live_returns)


In [ ]:


live_ret = None
comprehensive_overfitting_check(bt_ret, live_returns=live_ret, strategy_name="MyStrategy", show_plots=True)